# SAC Collector Head-to-Head — Sprint 1 (seed=7): ReBRAC β1∈{1.0, 4.0} sweep

**目标**：执行 [`docs/arrival_v2_sac_collector_design.md`](../docs/arrival_v2_sac_collector_design.md) **rev.3 §4.0.8 v2** Sprint 1 — ReBRAC β1∈{1.0, 4.0} × 4 tier × **seed 7** = **8 run**，作 SAC-collector head-to-head 的 ReBRAC 轴 (per-seed shard)。

**Seed-split rationale**：原 24-run sweep 按 seed 拆成 3 个 notebook (seed=42 / 0 / 7)，每份 8 run，可在 Colab Pro 多 session 并发开跑（理想 wall ~3.3h vs 串行 10-11h）。三份共同输出 24 run 与原协议完全等价。其它两份：[`_seed42`](./sac_collector_h2h_rebrac_sweep_seed42.ipynb) / [`_seed0`](./sac_collector_h2h_rebrac_sweep_seed0.ipynb) / [`_seed7`](./sac_collector_h2h_rebrac_sweep_seed7.ipynb)。

**前置 commits（已闭环）**：
- `97d394c` — SACCheckpointPolicy adapter (5 tests pass)
- `032b527` — Plan A 4 tier dataset collection completed
- `3cfd18a` — spec rev.3 §4.0.7 actual results land
- `5070862` — IPython shell magic refactor

**配置（spec §4.0.8 v2 锁定）**：

| 项 | 值 | 来源 |
|---|---|---|
| Algo | ReBRAC（vanilla critic，critic_LN=on） | FQL P2 primary baseline (= broad val v2 N0 anchor 同值) |
| β1 / β2 sweep | (β1=1.0/β2=2.0) + (β1=4.0/β2=2.0) | FQL P2 v1.4 §6.3 Q1b/Q1c (β1=1 dominate) + primary baseline (β1=4) |
| Datasets | 4 tier (random/medium/mexp/expert, step25k/425k/575k/600k) | Plan A 4 tier completed |
| Seed (本 shard) | **[7]** | 与 FQL P2 extended 3-seed paired (full set = [42, 0, 7]) |
| Total steps | 200_000 (uniform sampling) — FQL P2 §6.2 sister | FQL P2 v1.4 |
| batch / hidden / layers | 256 / 256 / 3 | 同上 |
| lr / γ / τ | 3e-4 / 0.99 / 0.005 | 同上 |
| Manifest (eval) | `benchmarks/single_u10_cross_tgt15.json` (100 ep, test_seed=456) | 与 FQL P2 paired |

**预算 (本 shard, seed=7)**：

| 阶段 | run | wallclock |
|---|---:|---:|
| Train batch 1: β1=1.0 × 4 tier × 1 seed | 4 | ~1.7h L4 |
| Train batch 2: β1=4.0 × 4 tier × 1 seed | 4 | ~1.7h L4 |
| evaluate_offline 100-ep × 8 run | 8 | ~17 min L4 |
| **Sprint 1 (seed=7) total** | **8** | **~3.3h L4** |

→ 1 个 Colab Pro L4 session 紧凑可完。三份 (seed=42/0/7) 同时启动 → 理想总 wall ~3.3h。

**Pre-registered finding 候选**（详 spec §4.0.8 v2 §3，全 24-run 完成后做 paired analysis）：
1. **β1 翻转点**：random/medium tier β1=4 ≥ β1=1（noisy data needs strong BC anchor）；expert tier β1=1 ≥ β1=4。
2. **(Deferred follow-up)** SAC vs rule-based collector — 不在本 sprint 1 cover；如 reviewer push back，用 N0 协议补 5h L4 sprint 1b。

## 0. 环境检查

In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

## 1. 挂载 Drive + cd 到项目根

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd

## 2. Sanity check — 4 dataset + manifest 存在

**预期**：5 项 ✅。任何 ⚠️ 都需停下定位（不要跳过 sanity 直接开跑 5h training）。

In [ ]:
from pathlib import Path

TIERS = [
    ("random", "step25k"),
    ("medium", "step425k"),
    ("mexp",   "step575k"),
    ("expert", "step600k"),
]

def dataset_dir(tier: str, step_tag: str) -> Path:
    return Path(f"offline_data/sac_{tier}_s0_h4_arrival_v2_re150_u10cross_seed46_{step_tag}_ep1000")

MANIFEST = Path("benchmarks/single_u10_cross_tgt15.json")

all_ok = True
for tier, step_tag in TIERS:
    d = dataset_dir(tier, step_tag)
    npz = d / "transitions.npz"
    if npz.exists():
        print(f"  ✅ {tier:10s}  {npz}  ({npz.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  ⚠️ {tier:10s}  {npz}  NOT FOUND")
        all_ok = False

print()
if MANIFEST.exists():
    print(f"  ✅ manifest  {MANIFEST}  ({MANIFEST.stat().st_size/1e3:.1f} KB)")
else:
    print(f"  ⚠️ manifest  {MANIFEST}  NOT FOUND — 用 generate_standard_benchmarks 先生成")
    all_ok = False

print()
if all_ok:
    print("✅ Sanity check passed — 可以开始 training.")
else:
    raise FileNotFoundError("Sanity check failed — 见上面 ⚠️ 项，定位后 rerun.")

## 3. Config matrix — 24 run plan

**Run identifier**：`rebrac_b1_{1p0|4p0}__{tier}__seed_{42|43|44}` — 与 spec §4.0.8 输出 schema 对齐。

**Skip-resume 协议**：per run 检查 `<save-dir>/trainer_state.json` 是否存在；存在则 skip（用于 Colab session crash recovery / 跨 session 接续）。

In [ ]:
BETA_CONFIGS = [
    ("b1_1p0", 1.0, 2.0),   # FQL P2 v1.4 Q1b/Q1c fix — actor β1=1, critic β2=2 锁 (dominate FQL on clean+noisy)
    ("b1_4p0", 4.0, 2.0),   # FQL P2 primary baseline (= broad val v2 N0 anchor 同值) 值
]
SEEDS = [7]  # per-seed shard; full set = [42, 0, 7] (seed42 / seed0 / seed7 notebooks)

CKPT_ROOT = Path("checkpoints/offline/sac_collector_h2h/rebrac")
RESULTS_ROOT = Path("results/offline/sac_collector_h2h/rebrac")

def save_dir(beta_tag: str, tier: str, seed: int) -> Path:
    return CKPT_ROOT / beta_tag / tier / f"seed_{seed}"

def test_result_path(beta_tag: str, tier: str, seed: int) -> Path:
    return RESULTS_ROOT / beta_tag / tier / f"seed_{seed}" / "test_result.json"

# Print plan + check skip-state (judged by agent_final.pt — written only at end of training,
# safer than trainer_state.json which may exist mid-train after a best-ckpt save)
n_total = 0
n_done = 0
for beta_tag, actor_pen, critic_pen in BETA_CONFIGS:
    for tier, step_tag in TIERS:
        for seed in SEEDS:
            n_total += 1
            sd = save_dir(beta_tag, tier, seed)
            done_marker = sd / "agent_final.pt"
            status = "✅ DONE" if done_marker.exists() else "⏳ TODO"
            if done_marker.exists():
                n_done += 1
            print(f"  [{status}]  {beta_tag} | {tier:10s} | seed={seed}  →  {sd}")
    print()

print(f"=== {n_done}/{n_total} runs already complete (skip-resume) ===")


## 4. Train batch 1 — ReBRAC β1=1.0 × 4 tier × 3 seed (12 run, ~5h L4)

**协议**（spec §4.0.8 v2）：`--actor-penalty-coef 1.0 --critic-penalty-coef 2.0`（β2 锁 2.0），其余与 batch 2 完全相同。

**Per-run skip-resume**：若 `<save-dir>/agent_final.pt` 已存在则跳过（train 结束才写，不会 false-positive）。Colab session crash 后重新运行本 cell 自动接续。

**实时输出**：训练用 `!python ...` IPython shell magic，stdout 直接流式 → 每 1000 step 的 `actor_loss/critic_loss` 实时滚动显示，不要等到最后才闪。

In [ ]:
import time
from pathlib import Path

t_start = time.time()

# Batch 1: β1=1.0
BATCH = BETA_CONFIGS[0]  # ("b1_1p0", 1.0, 2.0)
beta_tag, actor_pen, critic_pen = BATCH

for tier, step_tag in TIERS:
    for seed in SEEDS:
        sd = save_dir(beta_tag, tier, seed)
        sd_str = str(sd)
        npz = str(dataset_dir(tier, step_tag) / "transitions.npz")
        manifest_str = str(MANIFEST)
        if (sd / "agent_final.pt").exists():
            print(f"[skip-train] {beta_tag} | {tier} | seed={seed} (agent_final.pt exists)")
            continue
        sd.mkdir(parents=True, exist_ok=True)
        print(f"\n{'=' * 72}")
        print(f"  train {beta_tag} | {tier} | seed={seed}  →  {sd}")
        print(f"{'=' * 72}")
        t0 = time.time()
        !python -m scripts.train_offline \
            --algo rebrac \
            --offline-data '{npz}' \
            --manifest '{manifest_str}' \
            --probe-layout s0 \
            --history-length 4 \
            --task-geometry cross_stream \
            --target-speed 1.5 \
            --objective arrival_v2 \
            --sampling-mode uniform \
            --total-steps 200000 \
            --batch-size 256 \
            --hidden-dim 256 \
            --num-hidden-layers 3 \
            --actor-lr 3e-4 \
            --critic-lr 3e-4 \
            --gamma 0.99 \
            --tau 0.005 \
            --actor-penalty-coef {actor_pen} \
            --critic-penalty-coef {critic_pen} \
            --policy-noise 0.2 \
            --noise-clip 0.5 \
            --policy-freq 2 \
            --grad-clip-norm 10.0 \
            --normalizer-eps 1e-3 \
            --critic-layernorm \
            --no-actor-layernorm \
            --eval-every 0 \
            --skip-final-eval \
            --log-every 1000 \
            --seed {seed} \
            --device cuda \
            --save-dir '{sd_str}'
        print(f"[train done] {beta_tag} | {tier} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Batch 1 (β1={actor_pen}) complete, total = {(time.time() - t_start) / 60:.1f} min ===")


## 5. Train batch 2 — ReBRAC β1=4.0 × 4 tier × 3 seed (12 run, ~5h L4)

**协议**：`--actor-penalty-coef 4.0 --critic-penalty-coef 2.0`（FQL P2 primary baseline (= broad val v2 N0 anchor 同值) 值）。

**Checkpoint**：建议在 batch 1 全部 done 后再启动 batch 2，避免 Colab session timeout 中断时还要决定哪个 batch 接续。

In [ ]:
import time
from pathlib import Path

t_start = time.time()

# Batch 2: β1=4.0
BATCH = BETA_CONFIGS[1]  # ("b1_4p0", 4.0, 2.0)
beta_tag, actor_pen, critic_pen = BATCH

for tier, step_tag in TIERS:
    for seed in SEEDS:
        sd = save_dir(beta_tag, tier, seed)
        sd_str = str(sd)
        npz = str(dataset_dir(tier, step_tag) / "transitions.npz")
        manifest_str = str(MANIFEST)
        if (sd / "agent_final.pt").exists():
            print(f"[skip-train] {beta_tag} | {tier} | seed={seed} (agent_final.pt exists)")
            continue
        sd.mkdir(parents=True, exist_ok=True)
        print(f"\n{'=' * 72}")
        print(f"  train {beta_tag} | {tier} | seed={seed}  →  {sd}")
        print(f"{'=' * 72}")
        t0 = time.time()
        !python -m scripts.train_offline \
            --algo rebrac \
            --offline-data '{npz}' \
            --manifest '{manifest_str}' \
            --probe-layout s0 \
            --history-length 4 \
            --task-geometry cross_stream \
            --target-speed 1.5 \
            --objective arrival_v2 \
            --sampling-mode uniform \
            --total-steps 200000 \
            --batch-size 256 \
            --hidden-dim 256 \
            --num-hidden-layers 3 \
            --actor-lr 3e-4 \
            --critic-lr 3e-4 \
            --gamma 0.99 \
            --tau 0.005 \
            --actor-penalty-coef {actor_pen} \
            --critic-penalty-coef {critic_pen} \
            --policy-noise 0.2 \
            --noise-clip 0.5 \
            --policy-freq 2 \
            --grad-clip-norm 10.0 \
            --normalizer-eps 1e-3 \
            --critic-layernorm \
            --no-actor-layernorm \
            --eval-every 0 \
            --skip-final-eval \
            --log-every 1000 \
            --seed {seed} \
            --device cuda \
            --save-dir '{sd_str}'
        print(f"[train done] {beta_tag} | {tier} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Batch 2 (β1={actor_pen}) complete, total = {(time.time() - t_start) / 60:.1f} min ===")


## 6. Evaluate — 24 run × 100 ep test on `single_u10_cross_tgt15`

**Per-run skip-resume**：若 `<results-dir>/test_result.json` 已存在则跳过；若 `<save-dir>/agent_final.pt` 不存在则跳过并打 warn（train 未完）。

**预算**：24 × 2 min ≈ 50 min L4。

In [ ]:
import time

t_start_eval = time.time()

for beta_tag, _, _ in BETA_CONFIGS:
    for tier, _ in TIERS:
        for seed in SEEDS:
            sd = save_dir(beta_tag, tier, seed)
            sd_str = str(sd)
            out_path = test_result_path(beta_tag, tier, seed)
            out_path_str = str(out_path)
            manifest_str = str(MANIFEST)
            if out_path.exists():
                print(f"[skip-eval] {beta_tag} | {tier} | seed={seed} (test_result.json exists)")
                continue
            if not (sd / "agent_final.pt").exists():
                print(f"[warn] {beta_tag} | {tier} | seed={seed}  missing agent_final.pt — train not done?")
                continue
            out_path.parent.mkdir(parents=True, exist_ok=True)
            print(f"\n{'=' * 72}")
            print(f"  eval  {beta_tag} | {tier} | seed={seed}  →  {out_path}")
            print(f"{'=' * 72}")
            t0 = time.time()
            !python -m scripts.evaluate_offline \
                --checkpoint '{sd_str}' \
                --manifest '{manifest_str}' \
                --episodes 100 \
                --seed 456 \
                --device cuda \
                --output-json '{out_path_str}'
            print(f"[eval done] {beta_tag} | {tier} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Eval all complete, total = {(time.time() - t_start_eval) / 60:.1f} min ===")


## 7. Verify summary — 24 test_result.json → table

**输出**：
- per-(β, tier) mean ± std across 3 seeds (success_rate)
- β1=1 vs β1=4 per-tier delta（candidate β1 翻转点 finding）
- (Optional) FQL P2 v1.4 corresponding cell paired (expert↔E-uni / mexp↔M-uni-noise / medium↔M-multi-mix) — 详细 paired bootstrap CI 留给本地 sprint 1 analysis notebook

In [ ]:
import json
import numpy as np

results: dict[tuple, list[float]] = {}
missing = []
for beta_tag, _, _ in BETA_CONFIGS:
    for tier, _ in TIERS:
        seed_successes = []
        for seed in SEEDS:
            p = test_result_path(beta_tag, tier, seed)
            if not p.exists():
                missing.append((beta_tag, tier, seed))
                continue
            data = json.loads(p.read_text())
            sr = float(data.get("eval_success_rate", float("nan")))  # evaluate_offline.py outputs eval_success_rate in [0,1]
            seed_successes.append(sr)
        results[(beta_tag, tier)] = seed_successes

print("=== ReBRAC head-to-head on SAC tier datasets (test 100 ep × 3 seed) ===\n")
print(f"{'tier':12s} {'β1=1 (mean ± std)':25s} {'β1=4 (mean ± std)':25s} {'Δ (β1=4 − β1=1)':>18s}")
print("-" * 90)
for tier, _ in TIERS:
    s1 = results.get(("b1_1p0", tier), [])
    s4 = results.get(("b1_4p0", tier), [])
    m1 = float(np.mean(s1)) if s1 else float("nan")
    sd1 = float(np.std(s1, ddof=0)) if len(s1) > 1 else 0.0
    m4 = float(np.mean(s4)) if s4 else float("nan")
    sd4 = float(np.std(s4, ddof=0)) if len(s4) > 1 else 0.0
    delta = m4 - m1 if (s1 and s4) else float("nan")
    print(f"{tier:12s} {m1:.3f} ± {sd1:.3f}{'':>13} {m4:.3f} ± {sd4:.3f}{'':>13} {delta:+.3f}")

print()
if missing:
    print(f"⚠️ {len(missing)} runs missing test_result.json:")
    for m in missing:
        print(f"    {m}")
else:
    print("✅ All 24 runs have test_result.json — sprint 1 complete.")

## 8. Next steps（sprint 1 完成后）

1. **Paired bootstrap analysis（本地）**：把 24 test_result.json 与 **FQL P2 v1.4** ReBRAC β1∈{1,4} on E-uni/M-uni-noise/M-multi-mix 的 per-seed test 数字 paired，bootstrap 1000 resample CI for Δ_mechanism (SAC tier ↔ FQL P2 corresponding cell) per (tier, β1) — verify cross-noise-source universality。
2. **β1 翻转点判定**：Δ_β1 = mean(β1=4) - mean(β1=1) per tier。若 random/medium 显著 > 0 + expert 显著 < 0 → finding 候选 1 落地。
3. **Sprint 2 启动决策**：若 sprint 1 finding 1 已 strong（spread ≥ 0.1 + 3-seed std 不重叠）→ 启动 sprint 2 FQL × 12 run（~10h L4，1 session）补 finding 候选 2 (FQL ≈ ReBRAC β1=1 on clean expert)。
4. **结果回收**：本 notebook 跑完结果落 `_completed.ipynb` 副本 commit 进 git 作实验记录（参考 audit / collection completed 模式）。